---



# Unit 2 - Part 1a: LangChain Setup & Models

## 1. Introduction: Why LangChain?

Before we write code, you might ask: **"Why not just use the Gemini API directly?"**

Imagine you build an app using OpenAI. Six months later, you want to switch to Gemini because it's cheaper.
- **Without LangChain:** You have to rewrite all your API calls (different endpoint, different parameters, different response format).
- **With LangChain:** You change **one line of code**.

LangChain is a **framework** that provides a standard interface for any Language Model. It's like a universal adapter for AI.

## 2. Concept: Tokens (The Atom of AI)

Models don't read words. They read **Tokens**.
A token can be a word, part of a word, or even a space.

**Example:**
- Text: `"Hello World"`
- Tokens: `[101, 2055, 309]` (Hypothetical IDs)

### Cost & Context
You pay per token. The model has a limit on how many tokens it can remember (Context Window).

## 3. Setting Up the Environment

We need two main libraries:
1.  `langchain`: The core logic.
2.  `langchain-google-genai`: The specific connector for Google models.

Let's install them quietly.

In [ ]:
%pip install python-dotenv --upgrade --quiet langchain langchain-google-genai

## 4. Securely Loading API Keys

Never hardcode your API keys (e.g., `api_key = "AIzaSy..."`) in a notebook. If you share the notebook, your key is stolen.

**Best Practice:** Use `getpass` to enter it securely every session, or load it from environment variables.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

## 5. The Architecture (Flowchart)

What happens when we run the code below?

```mermaid
graph LR
    User[Your Python Code] -->|API Request (JSON)| Google[Google Gemini Cloud]
    Google -->|Inference| Model[Gemini 1.5 Pro]
    Model -->|Response| Google
    Google -->|API Response (JSON)| User
```

## 6. The `Temperature` Parameter (Critical Thinking)

When we initialize a model, we aren't just "turning it on". We are configuring its brain.

The most important setting is `temperature`.
- **Range:** 0.0 to 1.0 (sometimes higher).
- **Meaning:** How "random" should the choice of the next word be?

Let's create **two** versions of the same model to compare them side-by-side.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Model A: The "Accountant" (Precision)
llm_focused = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# Model B: The "Poet" (Creativity)
llm_creative = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=1.0)

## 7. Experiment: Consistency vs. Creativity

We will ask both models to "Define the word 'Idea' in one sentence."
We will run the code **TWICE** for each model.

**Hypothesis:**
- The Focused model (Temp=0) should say the *exact same thing* both times.
- The Creative model (Temp=1) should say *different things*.

In [ ]:
prompt = "Define the word 'Idea' in one sentence."

print("--- FOCUSED (Temp=0) ---")
print(f"Run 1: {llm_focused.invoke(prompt).content}")
print(f"Run 2: {llm_focused.invoke(prompt).content}")

--- FOCUSED (Temp=0) ---
Run 1: An idea is a thought, concept, or suggestion that is formed or exists in the mind.
Run 2: An idea is a thought, concept, or mental image formed in the mind.


In [ ]:
print("--- CREATIVE (Temp=1) ---")
print(f"Run 1: {llm_creative.invoke(prompt).content}")
print(f"Run 2: {llm_creative.invoke(prompt).content}")

--- CREATIVE (Temp=1) ---
Run 1: An idea is a thought, concept, or suggestion formulated in the mind, often serving as a starting point for action, understanding, or creation.
Run 2: An idea is a mental concept, thought, or plan that often serves as the basis for understanding, action, or creation.


## 8. Conclusion for Part 1a

**What did we learn?**
1.  **LangChain** abstracts the messy API details.
2.  **Tokens** are the currency of AI.
3.  **Temperature** is a control knob for randomness.

In the next notebook (**1b**), we will look at how to control the *Input* using Prompt Templates.

---



# Unit 2 - Part 1b: Prompts & Parsers

## 1. Introduction: The Pipeline

Real AI apps are not just `print(llm.invoke("hi"))`. They are pipelines.

### The Data Flow (Flowchart)
```mermaid
graph TD
    User[User Input 'Bob'] -->|Fill Template| Prompt[Prompt Object]
    Prompt -->|List of Messages| Model[LLM]
    Model -->|AIMessage Object| Parser[Output Parser]
    Parser -->|Clean String| Final['Hello Bob!']
```

In [ ]:
# Setup from Part 1a (Hidden for brevity)
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

## 2. Strings vs. Messages (Critical Thinking)

Most people start by talking to the AI like a human:
`llm.invoke("Translate this to French: Hello")`

But LLMs understand **Roles**:
- **System:** God-mode instructions. (e.g., "You are a calculator.")
- **Human:** The user.
- **AI:** The assistant.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# Scenario: Make the AI rude.
messages = [
    SystemMessage(content="You are a rude teenager. You use slang and don't care about grammar."),
    HumanMessage(content="What is the capital of France?")
]

response = llm.invoke(messages)
print(response.content)

Ugh, like, duh. It's Paris. Everyone knows that. What's wrong with you bro? So basic.


### Why System Messages matter?
If you just asked "What is the capital of France?" without the System Message, you'd get "Paris".
The System Message gives you **Control** over the personality and constraints.

## 3. The Context Window Concept

You might ask: "Can't I just paste a whole book into the System Message?"

Maybe.
Every model has a **Context Window** (e.g., 128k tokens for Gemini Flash).
- If you exceed it, the model **forgets the beginning**.
- It's like a sliding window over the conversation history.

## 4. Prompt Templates: The Safe Way

Don't do THIS:
`prompt = f"Translate {user_input} to Spanish"`

Do THIS:
`ChatPromptTemplate`.

It handles messy input (like quotes or newlines) safely.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Translate {input_language} to {output_language}."),
    ("human", "{text}")
])

# We can check what inputs it expects
print(f"Required variables: {template.input_variables}")

Required variables: ['input_language', 'output_language', 'text']


## 5. Output Parsers

Look at the output of `llm.invoke()`. It's an `AIMessage(content="...")`.
Usually, we just want the string inside.

**StrOutputParser** extracts just the text via regex or logic.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Raw Message
raw_msg = llm.invoke("Hi")
print(f"Raw Type: {type(raw_msg)}")

# Parsed String
clean_text = parser.invoke(raw_msg)
print(f"Parsed Type: {type(clean_text)}")
print(f"Content: {clean_text}")

Raw Type: <class 'langchain_core.messages.ai.AIMessage'>
Parsed Type: <class 'langchain_core.messages.base.TextAccessor'>
Content: Hello! How can I help you today?


## 6. Conclusion for Part 1b

We have the ingredients:
- **Model** (The Brain)
- **Prompt Template** (The Input Formatter)
- **Parser** (The Output Formatter)

In Part **1c**, we will chain them all together using **LCEL**.

---



# Unit 2 - Part 1c: LCEL (LangChain Expression Language)

## 1. Introduction: The Spaghetti Code Problem

In software engineering, doing things manually is often easy at first, but messy later.

We want to build a pipeline:
`Input -> Prompt -> Model -> Parser -> Output`

### Visualizing the Chain
```mermaid
graph LR
    Input({"topic": "Crows"}) -->|Inject| Template[ChatPromptTemplate]
    Template -->|Messages| LLM[Gemini Model]
    LLM -->|Message| Parser[StrOutputParser]
    Parser -->|String| Final["Crows are smart..."]
```

In [ ]:
# Setup (Hidden)
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
template = ChatPromptTemplate.from_template("Tell me a fun fact about {topic}.")
parser = StrOutputParser()

## 2. Method A: The Manual Way (Bad)

We call each step one by one. This is verbose and hard to modify.

In [ ]:
# Step 1: Format inputs
prompt_value = template.invoke({"topic": "Crows"})

# Step 2: Call Model
response_obj = llm.invoke(prompt_value)

# Step 3: Parse Output
final_text = parser.invoke(response_obj)

print(final_text)

Here's a fun one!

Crows are incredibly intelligent and have an amazing memory, especially when it comes to humans. They can **recognize individual human faces** and, if you've ever bothered them (like chasing them away from food or disturbing their nest), they might **hold a grudge against you!**

What's even wilder is that they can **teach other crows about "bad" human faces**, so one crow's enemy can become an entire murder's enemy for years! So, be nice to crows – they remember!


## 3. Method B: The LCEL Way (Good)

We use the **Pipe Operator (`|`)**.
It works just like Unix pipes: pass the output of the left side to the input of the right side.

In [ ]:
# Define the chain once
chain = template | llm | parser

# Invoke the whole chain
print(chain.invoke({"topic": "Octopuses"}))

Here's a fun one:

Octopuses have **three hearts!** Two pump blood through their gills, and the third circulates blood to the rest of their body. Talk about having a lot of love to give! ❤️❤️❤️


## 4. Why is this "Critical"? (Composability)

Imagine you want to swap the Model.
- **Manual:** You hunt for the line where `llm.invoke` happens.
- **LCEL:** You just change the `llm` variable in the chain definition.

Imagine you want to add a step (e.g., a spellchecker) between the prompt and the model.
- **LCEL:** `chain = template | spellchecker | llm | parser`

It makes your AI logic **Composable**.

## Assignment

Create a chain that:
1.  Takes a movie name.
2.  Asks for its release year.
3.  Calculates how many years ago that was (You can try just asking the LLM to do the math).

Try to do it in **one line of LCEL**.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

movie_chain = ChatPromptTemplate.from_template(
    "What year was the movie {movie} released? Also calculate how many years ago that was from 2026."
) | llm | StrOutputParser()

print(movie_chain.invoke({"movie": "Inception"}))
print(movie_chain.invoke({"movie": "KGF"}))


The movie Inception was released in **2010**.

From 2026, that will be **16 years** ago (2026 - 2010 = 16).
The movie KGF (Chapter 1) was released in **2018**.

From 2026, that was **8 years ago** (2026 - 2018 = 8).


Unit 2 - Part 2a: The Anatomy of a Prompt

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

# Using Low Temp for consistent comparison
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

In [ ]:
# The Task: Reject a candidate for a job.
task = "Write a rejection email to a candidate."

print("--- LAZY PROMPT ---")
print(llm.invoke(task).content)

--- LAZY PROMPT ---
Here are a few options for a rejection email, ranging from a standard, polite version to one that offers a bit more encouragement. Choose the one that best fits your company's culture and the stage of the hiring process.

---

**Option 1: Standard & Polite (Most Common)**

**Subject: Update on Your Application for [Job Title] at [Company Name]**

Dear [Candidate Name],

Thank you for your interest in the [Job Title] position at [Company Name] and for taking the time to [interview with our team / submit your application].

We appreciate you sharing your qualifications and experience with us. We received a large number of applications for this role, and after careful consideration, we have decided to move forward with other candidates whose qualifications and experience were a closer match for the specific requirements of this role at this time.

This was a very competitive search, and we truly appreciate the time and effort you invested in the process.

We wish you t

In [ ]:
structured_prompt = """
# Context
You are an HR Manager at a quirky startup called 'RocketBoots'.

# Objective
Write a rejection email to a candidate named Bob.

# Constraints
1. Be extremely brief (under 50 words).
2. Do NOT say 'we found someone better'. Say 'the role changed'.
3. Sign off with 'Keep flying'.

# Output Format
Plain text, no subject line.
"""

print("--- STRUCTURED PROMPT ---")
print(llm.invoke(structured_prompt).content)

--- STRUCTURED PROMPT ---
Hi Bob,

Thank you for your interest in RocketBoots.

We appreciate you taking the time to interview. However, the role's requirements have changed significantly since your application. We wish you the best in your job search.

Keep flying.


Assignment
Write a structured prompt to generate a Python Function.

Context: You are a Senior Python Dev.
Objective: Write a function to reverse a string.
Constraint: It must use recursion (no slicing [::-1]).
Style: Include detailed docstrings.


In [ ]:
structured_prompt = """
# Context
You are a Senior Python Developer.

# Objective
Write a Python function that reverses a string.

# Constraints
1. The function must use recursion.
2. Do NOT use slicing in any form (no s[1:], no [::-1]).
3. Do NOT use loops.
4. Use an index-based recursive approach.
5. Code must be clean and production-ready.

# Style
Include detailed docstrings explaining:
- Parameters
- Return value
- Base case
- Recursive step
- How recursion builds the reversed string

# Output Format
Only provide the Python code.
Do not include explanations outside the code.
"""

print(llm.invoke(structured_prompt).content)

```python
def reverse_string(s: str) -> str:
    """
    Reverses a string using an index-based recursive approach without loops or slicing.

    This function serves as a public interface, calling a private helper function
    to manage the recursive state (the current index).

    Args:
        s: The input string to be reversed.

    Returns:
        The reversed string.

    Raises:
        TypeError: If the input `s` is not a string.
    """
    if not isinstance(s, str):
        raise TypeError("Input must be a string.")

    def _reverse_recursive(current_string: str, index: int) -> str:
        """
        Helper function to recursively reverse a string using an index.

        Parameters:
            current_string: The original string (remains constant across recursive calls).
            index: The current index in the string being processed.

        Returns:
            The recursively built reversed string.

        Base Case:
            When 'index' equals the length of

Unit 2 - Part 2b: Zero-Shot to Few-Shot
1. Introduction: In-Context Learning


In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

In [ ]:
prompt_zero = "Combine 'Angry' and 'Hungry' into a funny new word."
print(f"Zero-Shot: {llm.invoke(prompt_zero).content}")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 31.146992819s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
prompt_few = """
Combine words into a funny new word. Give a sarcastic definition.

Input: Breakfast + Lunch
Output: Brunch (An excuse to drink alcohol before noon)

Input: Chill + Relax
Output: Chillax (What annoying people say when you are panic attacks)

Input: Angry + Hungry
Output:
"""
print(f"Few-Shot: {llm.invoke(prompt_few).content}")

Unit 2 - Part 2c: Advanced Templates & Theory

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 1. Our Database of Examples
examples = [
    {"input": "The internet is down.", "output": "We are observing connectivity latency."},
    {"input": "This code implies a bug.", "output": "The logic suggests unintended behavior."},
    {"input": "I hate this feature.", "output": "This feature does not align with my preferences."},
]

# 2. Template for ONE example
example_fmt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# 3. The Few-Shot Container
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_fmt,
    examples=examples
)

# 4. The Final Chain
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Corpo-Speak Translator. Rewrite the input to sound professional."),
    few_shot_prompt,      # Inject examples here
    ("human", "{text}")
])

chain = final_prompt | llm

print(chain.invoke({"text": "This app sucks."}).content)